# Standalone Inference Notebook — SimVPv2 PM2.5 Forecasting

**Purpose:** Load a trained SimVP checkpoint (`best_model.pt`) and generate `preds.npy` for submission.

**How to use on Kaggle:**
1. Add datasets: competition data, Kavin source dataset, and your checkpoint dataset.
2. Update `BEST_MODEL_PATH` in Cell 1.
3. Run all cells. `preds.npy` is written to `/kaggle/working/`.

**Model/Input convention:**
- SimVP input: `(B, T_in, C, H, W)`
- SimVP output: `(B, T_out, 1, H, W)`
- Submission format: `(B, H, W, T_out)` in physical PM2.5 units (µg/m³).

In [ ]:
# ── Cell 1: Bootstrap — paths, imports ─────────────────────────────────────
import os, sys
import numpy as np
import torch

KAGGLE_DATA_ROOT  = "/kaggle/input/datasets/khushisingh942004/aisehack"
KAVIN_DIR         = "/kaggle/input/datasets/sasidhar412/kavin-pm25-src/ANRF_AISEHack_Code/Kavin"

# Update this to your uploaded checkpoint dataset path
BEST_MODEL_PATH   = "/kaggle/input/kavin-best-model/best_model.pt"

OUT_DIR           = "/kaggle/working"
PRED_PATH         = os.path.join(OUT_DIR, "preds.npy")

if not os.path.exists("/kaggle"):
    KAVIN_DIR         = os.path.abspath("../Kavin")
    KAGGLE_DATA_ROOT  = os.path.abspath("../../aisehack-theme-2")
    BEST_MODEL_PATH   = os.path.abspath("../outputs/models/best_model.pt")
    OUT_DIR           = os.path.abspath("../outputs/submissions")
    PRED_PATH         = os.path.join(OUT_DIR, "preds.npy")

if KAVIN_DIR not in sys.path:
    sys.path.insert(0, KAVIN_DIR)

os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.exists(KAVIN_DIR), f"Source dir not found: {KAVIN_DIR}"
assert os.path.exists(KAGGLE_DATA_ROOT), f"Data root not found: {KAGGLE_DATA_ROOT}"
assert os.path.exists(BEST_MODEL_PATH), f"best_model.pt not found: {BEST_MODEL_PATH}"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"Data   : {KAGGLE_DATA_ROOT}")
print(f"Source : {KAVIN_DIR}")
print(f"Model  : {BEST_MODEL_PATH}")
print(f"Output : {PRED_PATH}")

In [ ]:
# ── Cell 2: Load config + bounds ────────────────────────────────────────────
from src.config import load_config
from src.data import load_minmax_bounds

cfg = load_config()
cfg['paths']['data'] = KAGGLE_DATA_ROOT
cfg['device'] = DEVICE
cfg['model']['type'] = 'simvpv2'
cfg['model']['simvp_in_channels'] = 24

bounds = load_minmax_bounds(cfg)
fmin_cpm, fmax_cpm = bounds['cpm25']
BASE_FEATURES = list(cfg['features']['base'])

print(f"cpm25 bounds: [{fmin_cpm}, {fmax_cpm}]")
print(f"Base features ({len(BASE_FEATURES)}): {BASE_FEATURES}")
print(f"Model type: {cfg['model']['type']}")

In [ ]:
# ── Cell 3: Build SimVP model and load checkpoint ───────────────────────────
from src.model import build_model

state = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model = build_model(cfg).to(DEVICE)
model.load_state_dict(state)
model.eval()

T_IN = int(cfg['time']['t_in'])
T_OUT = int(cfg['time']['t_out'])
N_TEST = int(cfg['data']['test_samples'])
H, W = 140, 124

params = sum(p.numel() for p in model.parameters())
print(f"Loaded SimVP checkpoint: {BEST_MODEL_PATH}")
print(f"Params: {params:,}")
print(f"T_in={T_IN}, T_out={T_OUT}, N_test={N_TEST}")

In [ ]:
# ── Cell 4: SimVP chunked preprocessing helper ──────────────────────────────
import gc
from src.data import build_simvp_test_chunk

BATCH_SIZE = min(int(cfg['training'].get('batch_size_test', 8)), 4)
print(f"Using inference batch size: {BATCH_SIZE}")

def build_test_chunk(start: int, end: int) -> np.ndarray:
    return build_simvp_test_chunk(
        cfg=cfg,
        bounds=bounds,
        feature_names=BASE_FEATURES,
        start=start,
        end=end,
    )

In [ ]:
# ── Cell 5: Smoke test one chunk ────────────────────────────────────────────
smoke_end = min(BATCH_SIZE, N_TEST)
x_smoke = build_test_chunk(0, smoke_end)
print(f"Smoke-test chunk shape : {x_smoke.shape}  (expected B,T,C,H,W)")
print(f"Smoke-test value range : [{x_smoke.min():.3f}, {x_smoke.max():.3f}]")
print(f"Smoke-test NaN/Inf     : {int(np.sum(~np.isfinite(x_smoke)))}")

del x_smoke
gc.collect()
print("Chunk builder sanity check passed.")

In [ ]:
# ── Cell 6: Streaming inference to disk-backed .npy memmap ──────────────────
from src.data import denormalize

TMP_PRED_PATH = os.path.join(OUT_DIR, 'preds_tmp.npy')
if os.path.exists(TMP_PRED_PATH):
    os.remove(TMP_PRED_PATH)
if os.path.exists(PRED_PATH):
    os.remove(PRED_PATH)

preds_mm = np.lib.format.open_memmap(
    TMP_PRED_PATH,
    mode='w+',
    dtype=np.float32,
    shape=(N_TEST, H, W, T_OUT),
)

model.eval()
with torch.no_grad():
    for i in range(0, N_TEST, BATCH_SIZE):
        j = min(i + BATCH_SIZE, N_TEST)
        x_batch = build_test_chunk(i, j)                         # (B,T,C,H,W)
        batch = torch.from_numpy(x_batch).to(DEVICE)

        pred = model(batch)
        # SimVP expected: (B,T,1,H,W)
        if pred.ndim == 5:
            if pred.shape[2] == 1:
                pred_norm = pred.squeeze(2).permute(0, 2, 3, 1).cpu().numpy()  # (B,H,W,T)
            else:
                pred_norm = pred.permute(0, 3, 4, 1, 2).squeeze(-1).cpu().numpy()
        elif pred.ndim == 4 and pred.shape[-1] == T_OUT:
            pred_norm = pred.cpu().numpy()
        elif pred.ndim == 4 and pred.shape[1] == T_OUT:
            pred_norm = pred.permute(0, 2, 3, 1).cpu().numpy()
        else:
            raise RuntimeError(f"Unexpected model output shape: {pred.shape}")

        pred_phys = denormalize(pred_norm, fmin_cpm, fmax_cpm, feat='cpm25')
        pred_phys = np.clip(pred_phys, 0.0, None).astype(np.float32)

        preds_mm[i:j] = pred_phys
        preds_mm.flush()

        if i == 0:
            print(f"[Batch 0] input {batch.shape} -> output {pred.shape}")
            print(f"          pred range: [{pred_phys.min():.1f}, {pred_phys.max():.1f}] ug/m3")
        print(f"Processed {j}/{N_TEST} samples", end='\r')

        del x_batch, batch, pred, pred_norm, pred_phys
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

print("\nInference complete.")
del preds_mm
os.replace(TMP_PRED_PATH, PRED_PATH)
print(f"Streaming predictions written to: {PRED_PATH}")

In [ ]:
# ── Cell 7: Verify saved preds.npy ──────────────────────────────────────────
loaded = np.load(PRED_PATH, mmap_mode='r')
assert loaded.shape == (N_TEST, H, W, T_OUT), f"Read-back shape mismatch: {loaded.shape}"

sample = np.asarray(loaded[: min(8, N_TEST)])
assert np.isfinite(sample).all(), "Sampled predictions contain NaN/Inf"

size_mb = os.path.getsize(PRED_PATH) / (1024**2)
print("=" * 55)
print(f"  preds.npy saved to : {PRED_PATH}")
print(f"  Shape              : {loaded.shape}")
print(f"  dtype              : {loaded.dtype}")
print(f"  File size          : {size_mb:.2f} MB")
print(f"  Sample range       : [{sample.min():.1f}, {sample.max():.1f}] ug/m3")
print(f"  Sample mean PM2.5  : {sample.mean():.2f} ug/m3")
print("=" * 55)
print("Ready to submit. Download preds.npy from /kaggle/working/ and upload it.")